In [1]:
import os
import pandas as pd
import numpy as np
import boto3
from tqdm import tqdm

#### Functions

In [2]:
def get_list_of_files_in_s3_location(cls_client, str_bucket_name, str_prefix):
    list_str_filename = []
    continuation_token = None

    while True:
        if continuation_token:
            dict_response = cls_client.list_objects_v2(
                Bucket=str_bucket_name,
                Prefix=str_prefix,
                ContinuationToken=continuation_token
            )
        else:
            dict_response = cls_client.list_objects_v2(
                Bucket=str_bucket_name,
                Prefix=str_prefix
            )

        # Add the filenames from this batch
        list_dict_contents = dict_response.get('Contents', [])
        for dict_contents in list_dict_contents:
            file_key = dict_contents['Key']
            if 'gzip' in file_key:
                filename = file_key
                list_str_filename.append(filename)

        # Check if there are more files to fetch
        continuation_token = dict_response.get('NextContinuationToken')
        if not continuation_token:
            break

    return list_str_filename

#### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

Project: 20241112-simple-model-test
Task: 05_create_df


#### Create output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Get list of files - tblDove

In [5]:
cls_client = boto3.client('s3')
list_str_filename_tbldove = get_list_of_files_in_s3_location(
    cls_client=cls_client,
    str_bucket_name=str_project,
    str_prefix='01_parse_payloads',
)
int_n_files_tbldove = len(list_str_filename_tbldove)
print(f'tblDove: {int_n_files_tbldove}')

tblDove: 855


#### Get list of files - Snowflake

In [6]:
cls_client = boto3.client('s3')
list_str_filename_snowflake = get_list_of_files_in_s3_location(
    cls_client=cls_client,
    str_bucket_name=str_project,
    str_prefix='02_parse_payloads',
)
int_n_files_snowflake = len(list_str_filename_snowflake)
print(f'Snowflake: {int_n_files_snowflake}')

Snowflake: 1135


#### Combine lists

In [7]:
list_str_filename = list_str_filename_tbldove + list_str_filename_snowflake
int_n_files = len(list_str_filename)
print(f'Total: {int_n_files}')

Total: 1990


#### Import accounts

In [8]:
list_cols = [
    'bigAccountId',
]
str_filename = 'df_targets.gzip'
str_uri = f's3://{str_project}/03_get_targets/01_classification/{str_filename}'
df = pd.read_parquet(
    str_uri,
    columns=list_cols,
)
# make sure its integer
df['bigAccountId'] = df['bigAccountId'].astype(int)
# make a list
list_int_account = list(df['bigAccountId'])
# rm dups
list_int_account = list(dict.fromkeys(list_int_account))
# get n accounts
int_n_accounts = len(list_int_account)
print(f'Accounts: {int_n_accounts}')
# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:279: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


Accounts: 339049


,bigAccountId
0,6
1,25
2,73
3,82
4,122
...,...
339044,8387997
339045,8388035
339046,8388060
339047,8388112


#### Loop through filenames

In [9]:
list_df = []
for str_filename in tqdm(list_str_filename):
    # get uri
    str_uri = f's3://{str_project}/{str_filename}'
    # import data
    df = pd.read_parquet(str_uri)
    # logic
    if 'ACCOUNTID' not in list(df.columns):
        dict_rename = {
            'bigAccountId': 'ACCOUNTID',
        }
        df.rename(columns=dict_rename, inplace=True)
    else:
        pass
    if 'REQUEST_DATETIME' not in list(df.columns):
        dict_rename = {
            'dtmCreatedDate': 'REQUEST_DATETIME',
        }
        df.rename(columns=dict_rename, inplace=True)
    else:
        pass
    # convert account id
    df['ACCOUNTID'] = df['ACCOUNTID'].astype(int)
    # subset to funded accounts
    df = df[df['ACCOUNTID'].isin(list_int_account)].copy()
    # append
    list_df.append(df)
    # save memory
    del df

100%|██████████| 1990/1990 [29:35<00:00,  1.12it/s]


#### Create data frame

In [10]:
%%time

df = pd.concat(list_df, ignore_index=True)
# save memory
del list_df
# show
df

CPU times: user 2min 30s, sys: 32.5 s, total: 3min 3s
Wall time: 3min 3s


,BIGDOVEID,ACCOUNTID,REQUEST_DATETIME,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,strcity__app,strname__app,...,sp81s__tu,sp82s__tu,sp83s__tu,sp84s__tu,sp85s__tu,sp86s__tu,sp87s__tu,sp99s__tu,fltfrontend__app,vehicleclass__app
0,588695.0,5712692,2021-07-26 16:15:47.424782700,1.0,5712692__7174945__20210719,5712692.0,7174945.0,1.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,588695.0,5712692,2021-07-26 16:15:47.424782700,0.0,5712692__7174946__20210719,5712692.0,7174946.0,0.0,Lostant,Illinois,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,588736.0,5702434,2021-07-26 16:29:29.390368600,1.0,5702434__7162486__20210707,5702434.0,7162486.0,1.0,CORALVILLE,Iowa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,588740.0,5713355,2021-07-26 16:31:38.506776200,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,588743.0,5713355,2021-07-26 16:32:04.321315800,1.0,5713355__7175762__20210719,5713355.0,7175762.0,1.0,Tooele,Utah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2927681,NaN,8385049,2024-11-12 23:10:01+00:00,0.0,0__0__20241109,8385049.0,10354345.0,0.0,POST FALLS,Idaho,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,17294.0,Class 2
2927682,NaN,8387703,2024-11-12 23:45:43+00:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927683,NaN,8387703,2024-11-13 02:50:14+00:00,1.0,0__0__20241111,8387703.0,10357680.0,1.0,Wooster,Ohio,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,23861.0,Class 1
2927684,NaN,8388035,2024-11-13 04:01:11+00:00,1.0,8388035103580741,8388035.0,10358074.0,1.0,nan,Maryland,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,24495.0,Class 2


#### Convert dtypes

In [11]:
for col in tqdm(df.columns):
    # get dtype
    str_dtype = df[col].dtype
    # logic
    if str_dtype not in ['int64','float64']:
        df[col] = df[col].astype(str)
    else:
        pass

100%|██████████| 3443/3443 [03:04<00:00, 18.70it/s] 


#### Save to s3

In [12]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(
    str_uri,
    compression='gzip',
)

CPU times: user 18min 13s, sys: 15.6 s, total: 18min 28s
Wall time: 18min 51s
